# 🚀 Semana 9-10: Proyectos Prácticos — Backend, Datos y APIs

**Curso:** Python, SQL, Ciencia de Datos y Análisis de Negocios  
**Duración estimada:** 2 semanas  
**Nivel:** Intermedio-Avanzado

---

## 📋 ¿Qué vas a construir esta semana?

| Proyecto | Descripción |
|----------|-------------|
| 🖥️ App web con backend | Flask + SQLAlchemy + autenticación básica |
| 📊 Pipeline de datos | Pandas + NumPy + matplotlib sobre dataset real |
| 🔌 API RESTful | CRUD completo documentado con ejemplos |
| 🤝 Open Source | Cómo contribuir a proyectos reales en GitHub |
| ⚡ Optimización | Perfilar y optimizar consultas lentas |

---

> 💡 **Semana de integración:**  
> Esta semana no hay teoría nueva — es práctica pura.  
> Vas a combinar todo lo aprendido en las semanas 1-8 en proyectos completos.

> **Instalaciones necesarias:**
> ```bash
> pip install flask sqlalchemy pandas numpy matplotlib requests
> ```

---
## 🖥️ Proyecto 1: Aplicación Web con Backend Completo

### 📘 Descripción

Vas a construir una **aplicación de gestión de tareas** (To-Do List) con:
- Backend en Flask
- Base de datos SQLite con SQLAlchemy
- Autenticación básica (usuarios con contraseña hasheada)
- API REST completa
- Pruebas con `test_client`

### 🏗️ Arquitectura

```
┌─────────────────────────────────────────┐
│              Flask App                   │
│                                          │
│  /auth/register  POST → crear usuario    │
│  /auth/login     POST → obtener token    │
│  /tareas         GET  → listar tareas    │
│  /tareas         POST → crear tarea      │
│  /tareas/<id>    PUT  → actualizar       │
│  /tareas/<id>    DELETE → eliminar       │
│                                          │
│         SQLAlchemy ORM                   │
│         ┌──────────┐  ┌──────────┐      │
│         │ usuarios │  │  tareas  │      │
│         └──────────┘  └──────────┘      │
└─────────────────────────────────────────┘
```

### 💡 Ejemplo resuelto — App completa de gestión de tareas

In [ ]:
from flask import Flask, request, jsonify
from sqlalchemy import create_engine, Column, Integer, String, Boolean, ForeignKey, text
from sqlalchemy.orm import DeclarativeBase, relationship, Session
from datetime import datetime
import hashlib
import json

# ── Modelos ──
class Base(DeclarativeBase): pass

class Usuario(Base):
    __tablename__ = 'usuarios'
    id       = Column(Integer, primary_key=True, autoincrement=True)
    username = Column(String(50), unique=True, nullable=False)
    password_hash = Column(String(64), nullable=False)
    tareas   = relationship('Tarea', back_populates='usuario', cascade='all, delete')

    @staticmethod
    def hashear(password):
        return hashlib.sha256(password.encode()).hexdigest()

    def verificar_password(self, password):
        return self.password_hash == self.hashear(password)

    def __repr__(self): return f'<Usuario({self.username})>'

class Tarea(Base):
    __tablename__ = 'tareas'
    id          = Column(Integer, primary_key=True, autoincrement=True)
    titulo      = Column(String(200), nullable=False)
    descripcion = Column(String(500), default='')
    completada  = Column(Boolean, default=False)
    prioridad   = Column(Integer, default=2)  # 1=alta, 2=media, 3=baja
    fecha_creacion = Column(String, default=lambda: datetime.now().isoformat())
    usuario_id  = Column(Integer, ForeignKey('usuarios.id'))
    usuario     = relationship('Usuario', back_populates='tareas')

    def to_dict(self):
        return {
            'id': self.id, 'titulo': self.titulo,
            'descripcion': self.descripcion, 'completada': self.completada,
            'prioridad': self.prioridad, 'fecha_creacion': self.fecha_creacion,
            'usuario_id': self.usuario_id
        }

# ── Configurar BD y App ──
engine = create_engine('sqlite:///:memory:', echo=False)
Base.metadata.create_all(engine)

app = Flask(__name__)

def get_usuario_from_header(request_obj):
    """Autenticación básica: header 'X-Username' + 'X-Password'."""
    username = request_obj.headers.get('X-Username')
    password = request_obj.headers.get('X-Password')
    if not username or not password:
        return None
    with Session(engine) as session:
        usuario = session.query(Usuario).filter_by(username=username).first()
        if usuario and usuario.verificar_password(password):
            session.expunge(usuario)
            return usuario
    return None

# ── Endpoints de autenticación ──
@app.route('/auth/register', methods=['POST'])
def register():
    datos = request.get_json()
    if not datos or not datos.get('username') or not datos.get('password'):
        return jsonify({'error': 'Se requieren username y password'}), 400
    with Session(engine) as session:
        if session.query(Usuario).filter_by(username=datos['username']).first():
            return jsonify({'error': 'Usuario ya existe'}), 409
        nuevo = Usuario(
            username=datos['username'],
            password_hash=Usuario.hashear(datos['password'])
        )
        session.add(nuevo)
        session.commit()
        return jsonify({'mensaje': f'Usuario {datos["username"]} creado', 'id': nuevo.id}), 201

@app.route('/auth/login', methods=['POST'])
def login():
    datos = request.get_json()
    with Session(engine) as session:
        usuario = session.query(Usuario).filter_by(username=datos.get('username','')).first()
        if usuario and usuario.verificar_password(datos.get('password','')):
            return jsonify({'mensaje': 'Login exitoso', 'username': usuario.username})
    return jsonify({'error': 'Credenciales inválidas'}), 401

# ── Endpoints de tareas ──
@app.route('/tareas', methods=['GET'])
def listar_tareas():
    usuario = get_usuario_from_header(request)
    if not usuario: return jsonify({'error': 'No autorizado'}), 401
    with Session(engine) as session:
        query = session.query(Tarea).filter_by(usuario_id=usuario.id)
        # Filtros opcionales
        if request.args.get('completada') == 'true':
            query = query.filter_by(completada=True)
        elif request.args.get('completada') == 'false':
            query = query.filter_by(completada=False)
        if request.args.get('prioridad'):
            query = query.filter_by(prioridad=int(request.args['prioridad']))
        tareas = query.order_by(Tarea.prioridad).all()
        return jsonify({'tareas': [t.to_dict() for t in tareas], 'total': len(tareas)})

@app.route('/tareas', methods=['POST'])
def crear_tarea():
    usuario = get_usuario_from_header(request)
    if not usuario: return jsonify({'error': 'No autorizado'}), 401
    datos = request.get_json()
    if not datos or not datos.get('titulo'):
        return jsonify({'error': 'Se requiere titulo'}), 400
    with Session(engine) as session:
        nueva = Tarea(
            titulo=datos['titulo'],
            descripcion=datos.get('descripcion', ''),
            prioridad=datos.get('prioridad', 2),
            usuario_id=usuario.id
        )
        session.add(nueva)
        session.commit()
        return jsonify(nueva.to_dict()), 201

@app.route('/tareas/<int:tarea_id>', methods=['PUT'])
def actualizar_tarea(tarea_id):
    usuario = get_usuario_from_header(request)
    if not usuario: return jsonify({'error': 'No autorizado'}), 401
    with Session(engine) as session:
        tarea = session.query(Tarea).filter_by(id=tarea_id, usuario_id=usuario.id).first()
        if not tarea: return jsonify({'error': 'Tarea no encontrada'}), 404
        datos = request.get_json()
        for campo in ['titulo', 'descripcion', 'completada', 'prioridad']:
            if campo in datos: setattr(tarea, campo, datos[campo])
        session.commit()
        return jsonify(tarea.to_dict())

@app.route('/tareas/<int:tarea_id>', methods=['DELETE'])
def eliminar_tarea(tarea_id):
    usuario = get_usuario_from_header(request)
    if not usuario: return jsonify({'error': 'No autorizado'}), 401
    with Session(engine) as session:
        tarea = session.query(Tarea).filter_by(id=tarea_id, usuario_id=usuario.id).first()
        if not tarea: return jsonify({'error': 'Tarea no encontrada'}), 404
        session.delete(tarea)
        session.commit()
        return jsonify({'mensaje': f'Tarea {tarea_id} eliminada'})

# ── Suite de pruebas completa ──
print('=' * 55)
print('   SUITE DE PRUEBAS — APP DE GESTIÓN DE TAREAS')
print('=' * 55)

with app.test_client() as client:
    headers_json = {'Content-Type': 'application/json'}

    # 1. Registrar usuarios
    print('\n[1] Registro de usuarios')
    for user, pwd in [('ana', 'pass123'), ('luis', 'secret456')]:
        r = client.post('/auth/register',
                        data=json.dumps({'username': user, 'password': pwd}),
                        headers=headers_json)
        print(f'  POST /auth/register ({user}) → {r.status_code}: {r.get_json()["mensaje"]}')

    # Registrar duplicado
    r = client.post('/auth/register',
                    data=json.dumps({'username': 'ana', 'password': 'otra'}),
                    headers=headers_json)
    print(f'  POST /auth/register (duplicado) → {r.status_code}: {r.get_json()["error"]}')

    # 2. Login
    print('\n[2] Login')
    r = client.post('/auth/login',
                    data=json.dumps({'username': 'ana', 'password': 'pass123'}),
                    headers=headers_json)
    print(f'  POST /auth/login (correcto) → {r.status_code}')
    r = client.post('/auth/login',
                    data=json.dumps({'username': 'ana', 'password': 'wrong'}),
                    headers=headers_json)
    print(f'  POST /auth/login (incorrecto) → {r.status_code}')

    # Headers de autenticación
    auth_ana  = {'X-Username': 'ana',  'X-Password': 'pass123', **headers_json}
    auth_luis = {'X-Username': 'luis', 'X-Password': 'secret456', **headers_json}

    # 3. Crear tareas
    print('\n[3] Crear tareas')
    tareas_ana = [
        {'titulo': 'Aprender Flask', 'prioridad': 1, 'descripcion': 'Completar notebook semana 5'},
        {'titulo': 'Practicar SQL',  'prioridad': 2, 'descripcion': 'Hacer ejercicios 3-4'},
        {'titulo': 'Leer documentación NumPy', 'prioridad': 3},
    ]
    for tarea in tareas_ana:
        r = client.post('/tareas', data=json.dumps(tarea), headers=auth_ana)
        t = r.get_json()
        print(f'  POST /tareas → {r.status_code}: [#{t["id"]}] {t["titulo"]} (prioridad {t["prioridad"]})')

    # 4. Listar tareas
    print('\n[4] Listar tareas de Ana')
    r = client.get('/tareas', headers=auth_ana)
    data_r = r.get_json()
    print(f'  GET /tareas → {r.status_code}: {data_r["total"]} tareas')
    for t in data_r['tareas']:
        print(f'    [#{t["id"]}] P{t["prioridad"]} — {t["titulo"]}')

    # 5. Marcar como completada
    print('\n[5] Marcar tarea #1 como completada')
    r = client.put('/tareas/1', data=json.dumps({'completada': True}), headers=auth_ana)
    print(f'  PUT /tareas/1 → {r.status_code}: completada={r.get_json()["completada"]}')

    # 6. Filtrar por estado
    print('\n[6] Filtrar por completada=false')
    r = client.get('/tareas?completada=false', headers=auth_ana)
    data_r = r.get_json()
    print(f'  GET /tareas?completada=false → {data_r["total"]} tareas pendientes')

    # 7. Aislamiento entre usuarios
    print('\n[7] Verificar aislamiento de datos')
    r_luis = client.get('/tareas', headers=auth_luis)
    r_ana  = client.get('/tareas', headers=auth_ana)
    print(f'  Luis ve {r_luis.get_json()["total"]} tareas (esperado: 0)')
    print(f'  Ana  ve {r_ana.get_json()["total"]}  tareas (esperado: 3)')

    # 8. Sin autenticación
    print('\n[8] Sin autenticación')
    r = client.get('/tareas')
    print(f'  GET /tareas (sin headers) → {r.status_code}: {r.get_json()["error"]}')


### ✏️ Ejercicio guiado 1 — Extender la app con categorías

Agregá una tabla `categorias` y relacionala con las tareas.

**Cambios a implementar:**
1. Nuevo modelo `Categoria` con: `id`, `nombre`, `color` (hex), `usuario_id`
2. Agregar `categoria_id` (FK opcional) en el modelo `Tarea`
3. Nuevos endpoints:
   - `GET /categorias` — listar categorías del usuario
   - `POST /categorias` — crear categoría
   - `GET /tareas?categoria_id=X` — filtrar tareas por categoría

**Pistas:**
- La relación es opcional: `categoria_id = Column(Integer, ForeignKey(...), nullable=True)`
- Al serializar la tarea en `to_dict()`, incluí el nombre de la categoría si existe
- Recordá que cada usuario solo puede ver sus propias categorías

In [ ]:
# ✏️ Extendé el código del ejemplo con categorías
# Copiá los modelos y endpoints anteriores y agregá los cambios aquí:

from flask import Flask, request, jsonify
from sqlalchemy import create_engine, Column, Integer, String, Boolean, ForeignKey
from sqlalchemy.orm import DeclarativeBase, relationship, Session
import json

class Base(DeclarativeBase): pass

# ✏️ Definí los modelos Usuario, Categoria y Tarea:


# ✏️ Configurá el engine y creá las tablas:


# ✏️ Implementá los endpoints:


# ✏️ Probá con test_client:


### 🚀 Proyecto libre 1 — Tu propia app web

Elegí uno de estos dominios y construí la app completa:

**Opción A — Blog personal:**  
Posts, comentarios, categorías, autores. Endpoints para publicar, editar y borrar posts.

**Opción B — Inventario de productos:**  
Productos, categorías, movimientos de stock (entrada/salida), alertas de stock bajo.

**Opción C — Sistema de turnos:**  
Servicios, horarios disponibles, reservas, cancelaciones, recordatorios.

**Requisitos mínimos para cualquier opción:**
1. Al menos 3 modelos relacionados con SQLAlchemy ORM
2. Autenticación básica (usuarios con contraseña hasheada)
3. CRUD completo para el recurso principal
4. Al menos 2 filtros en el listado
5. Suite de pruebas con `test_client` que cubra casos exitosos y errores

In [ ]:
from flask import Flask, request, jsonify
from sqlalchemy import create_engine, Column, Integer, String, Boolean, ForeignKey
from sqlalchemy.orm import DeclarativeBase, relationship, Session
import json

# 🚀 Tu app aquí:


---
## 📊 Proyecto 2: Pipeline de Análisis de Datos

### 📘 Descripción

Un **pipeline de datos** es una secuencia de pasos que transforma datos crudos en información útil:

```
Fuente  →  Carga  →  Limpieza  →  Transformación  →  Análisis  →  Visualización
  API        pandas    nulos,         nuevas            groupby,     matplotlib
  CSV        read_csv  tipos,         columnas          pivot,       seaborn
  SQL        read_sql  duplicados     derivadas         stats        charts
```

### 💡 Ejemplo resuelto — Pipeline completo de análisis de ventas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # para notebooks sin display
import matplotlib.pyplot as plt
from io import StringIO

# ✅ Pipeline completo: generación → limpieza → análisis → visualización

# ── PASO 1: Generar datos (simula carga desde CSV/API) ──
np.random.seed(42)
n = 1000

categorias = ['Electrónica', 'Ropa', 'Hogar', 'Deportes', 'Libros']
regiones   = ['AMBA', 'Centro', 'NOA', 'NEA', 'Cuyo', 'Patagonia']
vendedores = [f'Vendedor_{i:02d}' for i in range(1, 11)]

df_raw = pd.DataFrame({
    'fecha':      pd.date_range('2023-01-01', periods=n, freq='D').to_series().sample(n, replace=True).values,
    'categoria':  np.random.choice(categorias, n, p=[0.3,0.25,0.2,0.15,0.1]),
    'region':     np.random.choice(regiones, n),
    'vendedor':   np.random.choice(vendedores, n),
    'cantidad':   np.random.randint(1, 15, n),
    'precio':     np.round(np.random.uniform(100, 50000, n), 2),
    'devolucion': np.random.choice([True, False, None], n, p=[0.05, 0.90, 0.05]),
    'canal':      np.random.choice(['Online', 'Tienda', 'Telefono'], n, p=[0.5,0.35,0.15]),
})

# Introducir problemas de calidad para practicar limpieza
idx_nulos = np.random.choice(df_raw.index, 30)
df_raw.loc[idx_nulos[:15], 'precio']    = np.nan
df_raw.loc[idx_nulos[15:], 'categoria'] = np.nan
df_raw = pd.concat([df_raw, df_raw.sample(20)])  # duplicados

print('=== PASO 1: Datos crudos ===')
print(f'  Registros: {len(df_raw)}')
print(f'  Columnas:  {list(df_raw.columns)}')
print(f'  Nulos por columna:')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0].to_string())
print(f'  Duplicados: {df_raw.duplicated().sum()}')

# ── PASO 2: Limpieza ──
df = df_raw.copy()

# Eliminar duplicados
df = df.drop_duplicates()

# Imputar precios nulos con la mediana de su categoría
df['precio'] = df.groupby('categoria')['precio'].transform(
    lambda x: x.fillna(x.median())
)

# Imputar categorías nulas con 'Sin categoría'
df['categoria'] = df['categoria'].fillna('Sin categoría')

# Imputar devoluciones nulas como False
df['devolucion'] = df['devolucion'].fillna(False).astype(bool)

# Asegurar tipos correctos
df['fecha']    = pd.to_datetime(df['fecha'])
df['cantidad'] = df['cantidad'].astype(int)

print('\n=== PASO 2: Datos limpios ===')
print(f'  Registros tras limpieza: {len(df)}')
print(f'  Nulos restantes: {df.isnull().sum().sum()}')

# ── PASO 3: Transformaciones ──
df['ingreso']   = df['cantidad'] * df['precio'] * (~df['devolucion']).astype(int)
df['mes']       = df['fecha'].dt.to_period('M')
df['trimestre'] = df['fecha'].dt.to_period('Q')
df['dia_semana']= df['fecha'].dt.day_name()

print('\n=== PASO 3: Columnas derivadas añadidas ===')
print(f'  ingreso, mes, trimestre, dia_semana')

# ── PASO 4: Análisis ──
print('\n=== PASO 4: Análisis ===')

print('\nIngreso total por categoría:')
por_cat = df.groupby('categoria')['ingreso'].agg(['sum','mean','count'])
por_cat.columns = ['total', 'promedio', 'transacciones']
por_cat['participacion_%'] = (por_cat['total']/por_cat['total'].sum()*100).round(1)
print(por_cat.sort_values('total', ascending=False).to_string())

print('\nTop 3 vendedores por ingreso:')
top_vendedores = df.groupby('vendedor')['ingreso'].sum().nlargest(3)
for v, ing in top_vendedores.items():
    print(f'  {v}: ${ing:,.0f}')

print('\nTasa de devolución por canal:')
por_canal = df.groupby('canal')['devolucion'].agg(['sum','count'])
por_canal['tasa_%'] = (por_canal['sum']/por_canal['count']*100).round(2)
print(por_canal.to_string())

# Tendencia mensual con NumPy
mensual = df.groupby('mes')['ingreso'].sum()
x = np.arange(len(mensual))
coef = np.polyfit(x, mensual.values, 1)
print(f'\nTendencia mensual: ${coef[0]:+.0f} por mes ({"creciente ↑" if coef[0]>0 else "decreciente ↓"})')

# ── PASO 5: Reporte de texto ──
print('\n' + '='*50)
print('REPORTE EJECUTIVO')
print('='*50)
print(f'Período analizado:  {df["fecha"].min().date()} → {df["fecha"].max().date()}')
print(f'Total transacciones: {len(df):,}')
print(f'Ingreso total:       ${df["ingreso"].sum():>12,.0f}')
print(f'Ticket promedio:     ${df["ingreso"].mean():>12,.0f}')
print(f'Tasa de devolución:  {df["devolucion"].mean()*100:.1f}%')
print(f'Mejor categoría:     {por_cat["total"].idxmax()}')
print(f'Mejor canal:         {df.groupby("canal")["ingreso"].sum().idxmax()}')
print(f'Mejor vendedor:      {top_vendedores.index[0]}')

### ✏️ Ejercicio guiado 2 — Ampliar el pipeline con tabla pivote y anomalías

Usando el DataFrame `df` del ejemplo anterior, completá estos análisis.

**Pistas:**
- `df.pivot_table(values=, index=, columns=, aggfunc=)` para tablas cruzadas
- Un valor es anomalía si está más de 2 desviaciones estándar de la media
- `df.resample('W', on='fecha')['ingreso'].sum()` agrupa por semana

In [ ]:
# (Requiere haber ejecutado la celda anterior — df disponible)

# ✏️ Análisis 1: Tabla pivote región × categoría (ingresos)
print('=== Tabla pivote: Región × Categoría ===')
# pivot = df.pivot_table(...)
# print(pivot.round(0))


# ✏️ Análisis 2: Detección de anomalías en precio
print('\n=== Precios anómalos (> 2 desvíos estándar) ===')
# media = df['precio'].mean()
# std   = df['precio'].std()
# anomalias = df[abs(df['precio'] - media) > 2 * std]
# print(f'  Anomalías encontradas: {len(anomalias)}')
# print(anomalias[['categoria','precio','region']].head())


# ✏️ Análisis 3: Vendedor con mayor crecimiento entre Q1 y Q4
print('\n=== Crecimiento de vendedores Q1→Q4 ===')
# Pista: filtrá por trimestre, calculá ingresos por vendedor, comparalos


# ✏️ Análisis 4: Día de la semana con más ventas
print('\n=== Ventas por día de la semana ===')
# Pista: df.groupby('dia_semana')['ingreso'].sum()


<details>
<summary>👀 Ver solución</summary>

```python
# Análisis 1
pivot = df.pivot_table(values='ingreso', index='region',
                       columns='categoria', aggfunc='sum', fill_value=0)
print(pivot.round(0))

# Análisis 2
media, std = df['precio'].mean(), df['precio'].std()
anomalias = df[abs(df['precio'] - media) > 2 * std]
print(f'Anomalías: {len(anomalias)}')

# Análisis 3
q1 = df[df['trimestre'].astype(str).str.contains('Q1')].groupby('vendedor')['ingreso'].sum()
q4 = df[df['trimestre'].astype(str).str.contains('Q4')].groupby('vendedor')['ingreso'].sum()
crecimiento = ((q4 - q1) / q1 * 100).dropna().sort_values(ascending=False)
print(crecimiento.head(3))

# Análisis 4
orden_dias = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
por_dia = df.groupby('dia_semana')['ingreso'].sum().reindex(orden_dias)
print(por_dia)
```
</details>

### 🚀 Proyecto libre 2 — Pipeline con dataset real

Aplicá el pipeline completo sobre un dataset de tu elección.

**Fuentes de datos sugeridas:**
- [Kaggle Datasets](https://www.kaggle.com/datasets) — miles de datasets gratuitos
- [datos.gob.ar](https://datos.gob.ar/) — datos abiertos del gobierno argentino
- [Open Meteo](https://open-meteo.com/) — datos climáticos sin clave
- Dataset que uses en tu trabajo o estudio

**Tu pipeline debe incluir los 5 pasos:**
1. **Carga** — desde CSV, API o generado con NumPy
2. **Exploración** — `shape`, `info()`, `describe()`, nulos, duplicados
3. **Limpieza** — imputación, corrección de tipos, eliminación de outliers
4. **Análisis** — al menos 5 preguntas de negocio respondidas con código
5. **Reporte** — resumen ejecutivo en Markdown con tus conclusiones

In [ ]:
import pandas as pd
import numpy as np

# 🚀 Tu pipeline aquí:

# Paso 1: Carga


# Paso 2: Exploración


# Paso 3: Limpieza


# Paso 4: Análisis


# Paso 5: Reporte (continuá en la celda Markdown)


## 📝 Reporte ejecutivo

*✍️ Escribí aquí tus conclusiones del análisis:*

- **Dataset analizado:**
- **Período / alcance:**
- **Principales hallazgos:**
- **Recomendaciones:**

*(Doble click para editar)*

---
## 🔌 Proyecto 3: API RESTful Documentada

### 📘 Descripción

Una buena API necesita:
- Endpoints claros y consistentes
- Validación de datos de entrada
- Manejo de errores con mensajes útiles
- Documentación con ejemplos

### 📋 Convenciones REST

```
Recurso: /productos

GET    /productos           → listar (con filtros ?categoria=X&min_precio=Y)
GET    /productos/{id}      → obtener uno
POST   /productos           → crear
PUT    /productos/{id}      → reemplazar completo
PATCH  /productos/{id}      → actualizar parcial
DELETE /productos/{id}      → eliminar

Respuestas:
  200 OK           → GET, PUT, PATCH exitoso
  201 Created      → POST exitoso
  204 No Content   → DELETE exitoso
  400 Bad Request  → datos inválidos
  404 Not Found    → recurso no existe
  409 Conflict     → recurso duplicado
```

### 💡 Ejemplo resuelto — API de productos con validación completa

In [ ]:
from flask import Flask, request, jsonify
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey, text
from sqlalchemy.orm import DeclarativeBase, relationship, Session
import json

# ✅ API REST completa con validación y manejo de errores

class Base(DeclarativeBase): pass

class Categoria(Base):
    __tablename__ = 'categorias'
    id     = Column(Integer, primary_key=True, autoincrement=True)
    nombre = Column(String(50), nullable=False, unique=True)
    productos = relationship('Producto', back_populates='categoria')

class Producto(Base):
    __tablename__ = 'productos'
    id           = Column(Integer, primary_key=True, autoincrement=True)
    nombre       = Column(String(100), nullable=False)
    precio       = Column(Float, nullable=False)
    stock        = Column(Integer, default=0)
    sku          = Column(String(20), unique=True)  # código único
    categoria_id = Column(Integer, ForeignKey('categorias.id'))
    categoria    = relationship('Categoria', back_populates='productos')

    def to_dict(self, include_categoria=True):
        d = {'id': self.id, 'nombre': self.nombre, 'precio': self.precio,
             'stock': self.stock, 'sku': self.sku, 'categoria_id': self.categoria_id}
        if include_categoria and self.categoria:
            d['categoria'] = self.categoria.nombre
        return d

engine = create_engine('sqlite:///:memory:', echo=False)
Base.metadata.create_all(engine)

# Datos iniciales
with Session(engine) as s:
    cats = [Categoria(nombre=n) for n in ['Electrónica','Ropa','Hogar']]
    s.add_all(cats)
    s.flush()
    prods = [
        Producto(nombre='Laptop Pro', precio=1200.0, stock=10, sku='LAP-001', categoria_id=cats[0].id),
        Producto(nombre='Mouse USB',  precio=25.0,   stock=50, sku='MOU-001', categoria_id=cats[0].id),
        Producto(nombre='Camiseta',   precio=25.0,   stock=80, sku='CAM-001', categoria_id=cats[1].id),
    ]
    s.add_all(prods)
    s.commit()

app2 = Flask('productos_api')

def validar_producto(datos, parcial=False):
    """Valida los datos de un producto. parcial=True para PATCH."""
    errores = []
    if not parcial:
        if not datos.get('nombre'): errores.append('nombre es requerido')
        if datos.get('precio') is None: errores.append('precio es requerido')
    if 'precio' in datos and datos['precio'] is not None:
        if not isinstance(datos['precio'], (int, float)) or datos['precio'] <= 0:
            errores.append('precio debe ser un número positivo')
    if 'stock' in datos and datos['stock'] is not None:
        if not isinstance(datos['stock'], int) or datos['stock'] < 0:
            errores.append('stock debe ser un entero no negativo')
    return errores

@app2.route('/productos', methods=['GET'])
def listar_productos():
    with Session(engine) as session:
        query = session.query(Producto)
        if request.args.get('categoria'):
            cat = session.query(Categoria).filter_by(nombre=request.args['categoria']).first()
            if cat: query = query.filter_by(categoria_id=cat.id)
        if request.args.get('min_precio'):
            query = query.filter(Producto.precio >= float(request.args['min_precio']))
        if request.args.get('max_precio'):
            query = query.filter(Producto.precio <= float(request.args['max_precio']))
        if request.args.get('en_stock') == 'true':
            query = query.filter(Producto.stock > 0)
        productos = query.all()
        return jsonify({'data': [p.to_dict() for p in productos], 'total': len(productos)})

@app2.route('/productos/<int:pid>', methods=['GET'])
def obtener_producto(pid):
    with Session(engine) as session:
        p = session.get(Producto, pid)
        if not p: return jsonify({'error': f'Producto {pid} no encontrado'}), 404
        return jsonify(p.to_dict())

@app2.route('/productos', methods=['POST'])
def crear_producto():
    datos = request.get_json() or {}
    errores = validar_producto(datos)
    if errores: return jsonify({'error': 'Datos inválidos', 'detalle': errores}), 400
    with Session(engine) as session:
        if datos.get('sku') and session.query(Producto).filter_by(sku=datos['sku']).first():
            return jsonify({'error': f'SKU {datos["sku"]} ya existe'}), 409
        nuevo = Producto(nombre=datos['nombre'], precio=datos['precio'],
                         stock=datos.get('stock', 0), sku=datos.get('sku'),
                         categoria_id=datos.get('categoria_id'))
        session.add(nuevo)
        session.commit()
        return jsonify(nuevo.to_dict(include_categoria=False)), 201

@app2.route('/productos/<int:pid>', methods=['PATCH'])
def actualizar_producto(pid):
    datos = request.get_json() or {}
    errores = validar_producto(datos, parcial=True)
    if errores: return jsonify({'error': 'Datos inválidos', 'detalle': errores}), 400
    with Session(engine) as session:
        p = session.get(Producto, pid)
        if not p: return jsonify({'error': 'No encontrado'}), 404
        for campo in ['nombre','precio','stock','categoria_id','sku']:
            if campo in datos: setattr(p, campo, datos[campo])
        session.commit()
        return jsonify(p.to_dict(include_categoria=False))

@app2.route('/productos/<int:pid>', methods=['DELETE'])
def eliminar_producto(pid):
    with Session(engine) as session:
        p = session.get(Producto, pid)
        if not p: return jsonify({'error': 'No encontrado'}), 404
        session.delete(p)
        session.commit()
        return '', 204

@app2.route('/categorias', methods=['GET'])
def listar_categorias():
    with Session(engine) as session:
        cats = session.query(Categoria).all()
        return jsonify([{'id': c.id, 'nombre': c.nombre, 'productos': len(c.productos)} for c in cats])

# ── Documentación de la API ──
print('DOCUMENTACIÓN DE LA API')
print('=' * 55)
print('Base URL: http://localhost:5000')
print()
endpoints = [
    ('GET',    '/productos',         'Listar productos (?categoria=X&min_precio=Y&en_stock=true)'),
    ('GET',    '/productos/{id}',    'Obtener producto por ID'),
    ('POST',   '/productos',         'Crear producto {nombre, precio, stock?, sku?, categoria_id?}'),
    ('PATCH',  '/productos/{id}',    'Actualizar campos parciales'),
    ('DELETE', '/productos/{id}',    'Eliminar producto (204 No Content)'),
    ('GET',    '/categorias',        'Listar categorías con conteo de productos'),
]
for metodo, ruta, desc in endpoints:
    print(f'  {metodo:7} {ruta:25} → {desc}')

# ── Pruebas ──
print('\n' + '=' * 55)
print('PRUEBAS')
headers = {'Content-Type': 'application/json'}
with app2.test_client() as c:
    # Listar
    r = c.get('/productos')
    print(f'\nGET /productos → {r.status_code}: {r.get_json()["total"]} productos')
    # Filtrar
    r = c.get('/productos?categoria=Electrónica&min_precio=100')
    print(f'GET /productos?categoria=Electrónica&min_precio=100 → {r.get_json()["total"]} resultado(s)')
    # Crear válido
    r = c.post('/productos', data=json.dumps({'nombre':'Tablet','precio':299.0,'sku':'TAB-001','categoria_id':1}), headers=headers)
    print(f'POST /productos (válido) → {r.status_code}: id={r.get_json()["id"]}')
    # Crear inválido
    r = c.post('/productos', data=json.dumps({'nombre':'Test','precio':-50}), headers=headers)
    print(f'POST /productos (precio negativo) → {r.status_code}: {r.get_json()["detalle"]}')
    # SKU duplicado
    r = c.post('/productos', data=json.dumps({'nombre':'Otro','precio':100,'sku':'LAP-001'}), headers=headers)
    print(f'POST /productos (SKU duplicado) → {r.status_code}: {r.get_json()["error"]}')
    # PATCH
    r = c.patch('/productos/1', data=json.dumps({'precio':1099.0,'stock':8}), headers=headers)
    d = r.get_json()
    print(f'PATCH /productos/1 → {r.status_code}: precio={d["precio"]}, stock={d["stock"]}')
    # DELETE
    r = c.delete('/productos/3')
    print(f'DELETE /productos/3 → {r.status_code}')
    # Categorías
    r = c.get('/categorias')
    print(f'GET /categorias → {[c["nombre"]+":"+str(c["productos"]) for c in r.get_json()]}')

### 🚀 Proyecto libre 3 — Tu API documentada

Diseñá y construí una API REST para el dominio de tu elección, con:

1. **Al menos 2 recursos** relacionados (ej: autores y libros, usuarios y posts)
2. **CRUD completo** para el recurso principal
3. **Validación robusta** con mensajes de error claros
4. **Al menos 3 filtros** en el listado
5. **Documentación** impresa desde el código (como el ejemplo)
6. **Suite de pruebas** que cubra:
   - Casos exitosos (200, 201, 204)
   - Errores de validación (400)
   - Recursos no encontrados (404)
   - Conflictos (409)
   - Acceso no autorizado si implementás auth (401)

In [ ]:
from flask import Flask, request, jsonify
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey
from sqlalchemy.orm import DeclarativeBase, relationship, Session
import json

# 🚀 Tu API aquí:


---
## 🤝 Proyecto 4: Contribución a Código Abierto

### 📘 Cómo contribuir a un proyecto open source

Contribuir a proyectos reales es la mejor forma de aprender y construir tu portafolio.

#### Flujo de trabajo estándar

```bash
# 1. Fork del repositorio en GitHub (botón 'Fork')

# 2. Clonar tu fork
git clone https://github.com/TU_USUARIO/REPOSITORIO.git
cd REPOSITORIO

# 3. Crear rama para tu cambio
git checkout -b fix/nombre-del-bug
# o: git checkout -b feature/nueva-funcionalidad

# 4. Hacer los cambios y commitear
git add .
git commit -m 'fix: descripción clara del cambio'

# 5. Push a tu fork
git push origin fix/nombre-del-bug

# 6. Abrir Pull Request en GitHub
```

#### ¿Dónde encontrar issues para empezar?

| Etiqueta | Significado |
|----------|-------------|
| `good first issue` | Ideal para contribuidores nuevos |
| `beginner-friendly` | Pensado para principiantes |
| `help wanted` | El equipo necesita ayuda |
| `documentation` | Mejorar la documentación |
| `bug` | Error a corregir |

#### Proyectos Python amigables para empezar
- [pandas](https://github.com/pandas-dev/pandas/labels/good%20first%20issue)
- [requests](https://github.com/psf/requests/labels/good%20first%20issue)
- [scikit-learn](https://github.com/scikit-learn/scikit-learn/labels/good%20first%20issue)
- [Flask](https://github.com/pallets/flask/labels/good%20first%20issue)

### 💡 Ejemplo resuelto — Simular un flujo de contribución

In [ ]:
# ✅ Simulamos el proceso de encontrar y corregir un bug

# ─── CÓDIGO ORIGINAL (con bugs) ───
def calcular_estadisticas_v1(datos):
    """
    Calcula media, mediana y moda de una lista de números.
    BUG 1: No maneja listas vacías → ZeroDivisionError
    BUG 2: No maneja valores no numéricos → TypeError
    BUG 3: La mediana es incorrecta para listas de longitud par
    """
    media   = sum(datos) / len(datos)
    datos_ord = sorted(datos)
    n = len(datos_ord)
    mediana = datos_ord[n // 2]  # BUG: no promedia los dos centrales
    frecuencias = {}
    for x in datos:
        frecuencias[x] = frecuencias.get(x, 0) + 1
    moda = max(frecuencias, key=frecuencias.get)
    return {'media': media, 'mediana': mediana, 'moda': moda}

# ─── TESTS que descubren los bugs ───
print('=== Descubriendo bugs con tests ===')

# Test 1: lista normal
try:
    r = calcular_estadisticas_v1([1, 2, 3, 4, 5])
    print(f'  [1,2,3,4,5] → media={r["media"]}, mediana={r["mediana"]} (esperada: 3.0)')
except Exception as e:
    print(f'  ERROR: {e}')

# Test 2: longitud par (bug mediana)
try:
    r = calcular_estadisticas_v1([1, 2, 3, 4])
    print(f'  [1,2,3,4] → mediana={r["mediana"]} (esperada: 2.5) ← BUG')
except Exception as e:
    print(f'  ERROR: {e}')

# Test 3: lista vacía
try:
    r = calcular_estadisticas_v1([])
    print(f'  [] → {r}')
except ZeroDivisionError as e:
    print(f'  [] → ZeroDivisionError ← BUG')

# ─── CÓDIGO CORREGIDO (contribución) ───
def calcular_estadisticas_v2(datos):
    """
    Calcula media, mediana y moda de una lista de números.

    Args:
        datos: lista de números (int o float)
    Returns:
        dict con 'media', 'mediana', 'moda'
    Raises:
        ValueError: si la lista está vacía o contiene no-numéricos
    """
    if not datos:
        raise ValueError('La lista no puede estar vacía')
    if not all(isinstance(x, (int, float)) for x in datos):
        raise ValueError('Todos los elementos deben ser numéricos')

    media = sum(datos) / len(datos)

    datos_ord = sorted(datos)
    n = len(datos_ord)
    if n % 2 == 0:
        mediana = (datos_ord[n//2 - 1] + datos_ord[n//2]) / 2  # FIX
    else:
        mediana = datos_ord[n // 2]

    frecuencias = {}
    for x in datos:
        frecuencias[x] = frecuencias.get(x, 0) + 1
    moda = max(frecuencias, key=frecuencias.get)

    return {'media': media, 'mediana': mediana, 'moda': moda}

print('\n=== Tests sobre la versión corregida ===')
casos = [
    ([1, 2, 3, 4, 5],        {'media': 3.0, 'mediana': 3.0, 'moda': 1}),
    ([1, 2, 3, 4],           {'media': 2.5, 'mediana': 2.5, 'moda': 1}),
    ([2, 2, 3, 4, 4, 4, 5], {'media': None, 'mediana': 4.0, 'moda': 4}),
]
for datos, esperado in casos:
    resultado = calcular_estadisticas_v2(datos)
    mediana_ok = resultado['mediana'] == esperado['mediana']
    moda_ok    = resultado['moda']    == esperado['moda']
    print(f'  {str(datos):30} mediana={'✅' if mediana_ok else '❌'} moda={'✅' if moda_ok else '❌'}')

# Test errores esperados
print('\n=== Tests de manejo de errores ===')
for caso_error in [[], [1, 'a', 3]]:
    try:
        calcular_estadisticas_v2(caso_error)
        print(f'  {caso_error} → No lanzó error ❌')
    except ValueError as e:
        print(f'  {caso_error} → ValueError: {e} ✅')

### 🚀 Proyecto libre 4 — Contribución simulada

Simulá un proceso completo de contribución open source:

1. **Elegí una función** de las semanas anteriores que tenga al menos 2 bugs o limitaciones
2. **Escribí tests** que demuestren los problemas (deben fallar con el código original)
3. **Corregí el código** hasta que todos los tests pasen
4. **Documentá los cambios** con docstrings claros y ejemplos
5. **Escribí el mensaje de commit** siguiendo el estándar:
   - `fix: descripción del bug corregido`
   - `feat: descripción de la nueva funcionalidad`
   - `docs: mejora en la documentación`
   - `test: nuevos tests agregados`

**Bonus:** Buscá un issue real en GitHub etiquetado `good first issue` en un proyecto Python y describí cómo lo resolverías.

In [ ]:
# 🚀 Tu contribución simulada aquí:

# 1. Código original con bugs:


# 2. Tests que descubren los bugs:


# 3. Código corregido:


# 4. Tests que verifican la corrección:


# Mensaje de commit: 'fix: ...'


---
## ⚡ Proyecto 5: Perfilado y Optimización

### 📘 Proceso de optimización

```
1. MEDIR primero (nunca optimizar a ciegas)
   ↓
2. IDENTIFICAR el cuello de botella
   ↓
3. OPTIMIZAR solo lo que importa
   ↓
4. VERIFICAR que la optimización funciona
   ↓
5. DOCUMENTAR el cambio y la mejora
```

#### Herramientas de Python
```python
import time
import cProfile
import timeit

# Medir una función específica
%timeit mi_funcion(datos)   # en Jupyter

# Perfilar línea a línea
cProfile.run('mi_funcion(datos)')
```

#### Técnicas de optimización SQL
```sql
-- ANTES: sin índice, escanea toda la tabla
SELECT * FROM ventas WHERE cliente_id = 42;

-- DESPUÉS: con índice, acceso directo
CREATE INDEX idx_ventas_cliente ON ventas(cliente_id);
SELECT * FROM ventas WHERE cliente_id = 42;
```

### 💡 Ejemplo resuelto — Identificar y corregir cuellos de botella

In [ ]:
import time
import sqlite3
import random
import cProfile
import pstats
import io

# ✅ Proceso completo de optimización

# ── PASO 1: Código lento (a optimizar) ──
def procesar_ventas_lento(ventas):
    """
    Calcula estadísticas por categoría.
    PROBLEMAS:
    - Bucles Python puros en vez de operaciones vectoriales
    - Múltiples pasadas sobre los datos
    - Usa lista cuando debería usar dict/set
    """
    categorias = []
    for v in ventas:
        if v['categoria'] not in categorias:  # O(n) por búsqueda en lista
            categorias.append(v['categoria'])

    resultado = {}
    for cat in categorias:
        total = 0
        count = 0
        for v in ventas:             # O(n) por cada categoría = O(n*k)
            if v['categoria'] == cat:
                total += v['monto']
                count += 1
        resultado[cat] = {'total': total, 'promedio': total/count if count else 0, 'count': count}
    return resultado

def procesar_ventas_rapido(ventas):
    """
    Versión optimizada: una sola pasada O(n).
    - Un solo recorrido de los datos
    - Usa defaultdict para acumulación directa
    """
    from collections import defaultdict
    totales = defaultdict(float)
    conteos = defaultdict(int)

    for v in ventas:           # UNA sola pasada
        cat = v['categoria']
        totales[cat] += v['monto']
        conteos[cat] += 1

    return {
        cat: {'total': totales[cat],
              'promedio': totales[cat]/conteos[cat],
              'count': conteos[cat]}
        for cat in totales
    }

# Generar datos de prueba
cats = ['Electrónica','Ropa','Hogar','Deportes','Libros']
ventas = [
    {'categoria': random.choice(cats), 'monto': random.uniform(10, 5000)}
    for _ in range(50_000)
]

# ── PASO 2: Medir ──
print('=== Comparación de rendimiento ===')
n_runs = 5
t0 = time.perf_counter()
for _ in range(n_runs): procesar_ventas_lento(ventas)
t_lento = (time.perf_counter() - t0) / n_runs

t0 = time.perf_counter()
for _ in range(n_runs): procesar_ventas_rapido(ventas)
t_rapido = (time.perf_counter() - t0) / n_runs

print(f'  Versión lenta:  {t_lento*1000:>8.1f} ms')
print(f'  Versión rápida: {t_rapido*1000:>8.1f} ms')
print(f'  Mejora:         {t_lento/t_rapido:>8.1f}x más rápido')

# Verificar que producen el mismo resultado
r1 = procesar_ventas_lento(ventas)
r2 = procesar_ventas_rapido(ventas)
iguales = all(abs(r1[c]['total'] - r2[c]['total']) < 0.01 for c in r1)
print(f'  Resultados iguales: {'✅' if iguales else '❌'}')

# ── PASO 3: Optimización SQL con índices ──
print('\n=== Optimización SQL ===')

def benchmark_sql(n=100_000, con_indice=False):
    conn = sqlite3.connect(':memory:')
    conn.execute('CREATE TABLE ventas (id INTEGER, cliente_id INTEGER, monto REAL, categoria TEXT)')
    datos = [(i, random.randint(1,1000), random.uniform(10,5000), random.choice(cats))
             for i in range(n)]
    conn.executemany('INSERT INTO ventas VALUES (?,?,?,?)', datos)
    if con_indice:
        conn.execute('CREATE INDEX idx_cliente ON ventas(cliente_id)')
        conn.execute('CREATE INDEX idx_categoria ON ventas(categoria)')
    conn.commit()

    queries = [
        ('Buscar por cliente', 'SELECT * FROM ventas WHERE cliente_id = 42'),
        ('Agrupar por cat',    'SELECT categoria, SUM(monto) FROM ventas GROUP BY categoria'),
    ]
    for nombre, q in queries:
        t0 = time.perf_counter()
        conn.execute(q).fetchall()
        t = (time.perf_counter()-t0)*1000
        tipo = 'CON índice' if con_indice else 'SIN índice'
        print(f'  {tipo} | {nombre:20} | {t:.2f} ms')
    conn.close()

benchmark_sql(con_indice=False)
benchmark_sql(con_indice=True)

### 🚀 Proyecto libre 5 — Optimización de consultas

Tomá la aplicación que construiste en el Proyecto 1 (o el Pipeline del Proyecto 2) y:

1. **Identificá** las 3 operaciones más lentas usando `time.perf_counter()`
2. **Optimizá** cada una:
   - Reemplazá bucles Python con operaciones de pandas/NumPy donde corresponda
   - Agregá índices SQL a columnas usadas en `WHERE` y `JOIN`
   - Eliminá queries dentro de bucles (N+1 problem)
3. **Documentá** la mejora con tabla comparativa antes/después
4. **Asegurate** de que los resultados sigan siendo correctos después de optimizar

In [ ]:
import time

# 🚀 Tu proceso de optimización aquí:

# 1. Identificar cuellos de botella:


# 2. Optimizar:


# 3. Tabla comparativa:


# 4. Verificar correctitud:


---
## ✅ Resumen de la Semana 9-10

### Lo que construiste

| Proyecto | Habilidades practicadas |
|----------|--------------------------|
| App web | Flask + SQLAlchemy ORM + autenticación + pruebas |
| Pipeline de datos | Pandas + NumPy + limpieza + análisis + reporte |
| API RESTful | Validación + manejo de errores + documentación |
| Open Source | Testing + debugging + convenciones de contribución |
| Optimización | Perfilado + índices SQL + operaciones vectoriales |

### ➡️ Próximos pasos — Semana 11-12
- Frameworks web: Django (completo) vs Flask (micro)
- SQLAlchemy avanzado: migraciones con Alembic
- APIs externas: OAuth, rate limiting, manejo de errores de red

---

### 📚 Recursos recomendados
- [Flask — Documentación oficial](https://flask.palletsprojects.com/)
- [SQLAlchemy — ORM Tutorial](https://docs.sqlalchemy.org/en/20/orm/quickstart.html)
- [First Contributions](https://github.com/firstcontributions/first-contributions) — guía práctica
- [Real Python — Flask Tutorial](https://realpython.com/flask-project/)
- [Python Profiling](https://docs.python.org/3/library/profile.html)